# Cell 1: Imports and Environment Setup

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import custom SAHI wrapper components
from sahi_inference import SahiInference, load_config, CLASS_NAMES

print("✅ Environment and imports ready.")

# Cell 2: Define Model Paths and Directories

In [ ]:
# 1. Define the 3 models to benchmark (UPDATE PATHS TO YOUR CHECKPOINTS)
MODELS = {
    "surgical_early": "/home/lamacpp/Documents/car_defect_detection/mlruns/1/4a4675e34e8847539056b2c2d9d96918/artifacts/weights/best.pt"
}

# 2. Define Dataset Directories
DEFECT_TEST_DIR = "/home/lamacpp/Documents/car_defect_detection/data/processed/yolo_seg/images/test"
CLEAN_EVAL_DIR = "/home/lamacpp/Documents/car_defect_detection/data/processed/clean_cars/images/clean_eval"

# 3. SAHI Configuration
SAHI_CONFIG_PATH = "/home/lamacpp/Documents/car_defect_detection/configs/inference/sahi_production.yaml"

# Verify paths exist
for name, path in MODELS.items():
    if not os.path.exists(path):
        print(f"⚠️ Warning: Model weights not found for {name} at {path}")
        
print(f"✅ Found {len(glob.glob(os.path.join(DEFECT_TEST_DIR, '*.*')))} defect test images.")
print(f"✅ Found {len(glob.glob(os.path.join(CLEAN_EVAL_DIR, '*.*')))} clean eval images.")

# Cell 3: SAHI Inference Runner


In [ ]:
def get_image_paths(directory):
    """Helper to gather all image paths from a directory."""
    exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp"]
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(directory, ext)))
    return sorted(paths)

def run_sahi_benchmark(model_name, model_path, image_paths, cfg):
    """Runs SAHI inference on a list of images using the custom wrapper."""
    print(f"🔄 Loading model: {model_name}...")
    # Initialize engine (defaults to cuda:0, change if needed)
    engine = SahiInference(model_path, cfg, device="cuda:0") 
    
    results = []
    for img_path in tqdm(image_paths, desc=f"SAHI Inference ({model_name})", leave=False):
        preds = engine.predict(img_path)
        results.append({
            "image_path": img_path,
            "image_id": Path(img_path).stem,
            "predictions": preds
        })
    return results

# Cell 4: Execute Inference on Both Datasets


In [ ]:
# Load SAHI config once
cfg = load_config(SAHI_CONFIG_PATH)

defect_images = get_image_paths(DEFECT_TEST_DIR)
clean_images = get_image_paths(CLEAN_EVAL_DIR)

all_results = {}

for model_name, model_path in MODELS.items():
    if not os.path.exists(model_path):
        continue
        
    print(f"\n{'='*20} BENCHMARKING: {model_name} {'='*20}")
    all_results[model_name] = {
        "defect_preds": run_sahi_benchmark(model_name, model_path, defect_images, cfg),
        "clean_preds": run_sahi_benchmark(model_name, model_path, clean_images, cfg)
    }
    
print("\n🎉 All inference runs complete!")

# Cell 5: Calculate Metrics (FPR & Hallucination Breakdown)


In [ ]:
def calculate_metrics(model_name, defect_preds, clean_preds):
    metrics = {"Model": model_name}
    
    # --- 1. Clean Image False Positive Rate (FPR) ---
    clean_images_with_fp = 0
    total_clean_images = len(clean_preds)
    fp_class_counts = {name: 0 for name in CLASS_NAMES.values()}
    total_fp = 0
    
    for res in clean_preds:
        if len(res["predictions"]) > 0:
            clean_images_with_fp += 1
            for p in res["predictions"]:
                fp_class_counts[p["name"]] += 1
                total_fp += 1
                
    metrics["Clean_Images_Total"] = total_clean_images
    metrics["Clean_Images_with_FP"] = clean_images_with_fp
    metrics["Clean_FPR (%)"] = round((clean_images_with_fp / total_clean_images * 100), 2) if total_clean_images > 0 else 0.0
    metrics["Total_False_Positives"] = total_fp
    metrics["Most_Hallucinated_Class"] = max(fp_class_counts, key=fp_class_counts.get) if any(fp_class_counts.values()) else "None"
    metrics["Max_Hallucinated_Count"] = max(fp_class_counts.values()) if any(fp_class_counts.values()) else 0
    
    # --- 2. Defect Inference Stats ---
    total_defect_images = len(defect_preds)
    images_with_defects = sum(1 for res in defect_preds if len(res["predictions"]) > 0)
    total_detections = sum(len(res["predictions"]) for res in defect_preds)
    
    metrics["Defect_Images_Total"] = total_defect_images
    metrics["Images_With_Detections"] = images_with_defects
    metrics["Total_Detections"] = total_detections
    
    return metrics

# Generate Summary Table
metrics_list = []
for model_name, res in all_results.items():
    m = calculate_metrics(model_name, res["defect_preds"], res["clean_preds"])
    metrics_list.append(m)

df_metrics = pd.DataFrame(metrics_list)
print(df_metrics)

# Cell 6: Side-by-Side Visualization Function


In [ ]:
def plot_side_by_side_comparison(image_paths, all_results, dataset_type, max_images=3):
    """Plots side-by-side predictions for the given image paths across all models."""
    num_models = len(all_results)
    num_images = min(len(image_paths), max_images)
    
    fig, axes = plt.subplots(num_images, num_models + 1, figsize=(6 * (num_models + 1), 6 * num_images))
    if num_images == 1:
        axes = [axes]
        
    model_names = list(all_results.keys())
    
    for i in range(num_images):
        img_path = image_paths[i]
        img_name = Path(img_path).stem
        
        # Load base image
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Plot Original
        axes[i][0].imshow(img_rgb)
        axes[i][0].set_title(f"Original\n{img_name}", fontsize=14, fontweight='bold')
        axes[i][0].axis("off")
        
        # Plot Model Predictions
        for j, model_name in enumerate(model_names):
            # Find predictions for this image
            preds_key = "defect_preds" if dataset_type == "defect" else "clean_preds"
            preds = next((r["predictions"] for r in all_results[model_name][preds_key] if r["image_id"] == img_name), [])
            
            img_copy = img_rgb.copy()
            for p in preds:
                x1, y1, x2, y2 = p["bbox"]
                
                # Draw mask contour if available
                if "mask" in p and p["mask"] is not None:
                    mask = p["mask"].astype(np.uint8) * 255
                    # Resize mask to image dimensions if necessary
                    if mask.shape[:2] != img_copy.shape[:2]:
                        mask = cv2.resize(mask, (img_copy.shape[1], img_copy.shape[0]), interpolation=cv2.INTER_NEAREST)
                        
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(img_copy, contours, -1, (255, 0, 0), 2)
                else:
                    # Fallback to bounding box
                    cv2.rectangle(img_copy, (x1, y1), (x2, y2), (255, 0, 0), 2)
                
                # Label
                label = f"{p['name']} {p['score']:.2f}"
                cv2.putText(img_copy, label, (x1, max(0, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
                
            axes[i][j+1].imshow(img_copy)
            
            # Color code title based on dataset type and detections
            title_color = "black"
            if dataset_type == "clean" and len(preds) > 0:
                title_color = "red" # Highlight False Positives
                
            axes[i][j+1].set_title(f"{model_name}\n({len(preds)} detections)", fontsize=14, color=title_color, fontweight='bold')
            axes[i][j+1].axis("off")
            
    plt.tight_layout()
    plt.show()

# Cell 7: Visualize Defect Test Set (True Positives / Recall Check)


In [ ]:
# Select a few sample images from the defect test set
sample_defect_images = get_image_paths(DEFECT_TEST_DIR)[:3]

print("\n--- 🔍 Side-by-Side Visual Comparison (Defect Test Set) ---")
if sample_defect_images:
    plot_side_by_side_comparison(sample_defect_images, all_results, dataset_type="defect", max_images=3)
else:
    print("No defect test images found to visualize.")

# Cell 8: Visualize Clean Car Eval Set (False Positive / Hallucination Check)


In [ ]:
# Select sample clean images
sample_clean_images = get_image_paths(CLEAN_EVAL_DIR)[:3]

print("\n--- 🧼 Side-by-Side Visual Comparison (Clean Car Eval Set - False Positives) ---")
print("⚠️ Note: Any detections on these images are FALSE POSITIVES (Hallucinations). Model titles will be red if they hallucinate.")
if sample_clean_images:
    plot_side_by_side_comparison(sample_clean_images, all_results, dataset_type="clean", max_images=3)
else:
    print("No clean eval images found to visualize.")